In [2]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error 

def prepare_data_and_train_model_a(input_filename):
    df = pd.read_csv(input_filename)
    df_filtered = df[
        (df['Region'] == 'Asia') & 
        (df['Fuel_Type'].isin(['Electric', 'Hybrid']))
    ].copy()
    
    if df_filtered.empty:
        print("Error: No electric/hybrid data found for Asia.")
        return None
    df_annual_avg = df_filtered.groupby('Year')['Price_USD'].mean().reset_index()
    df_annual_avg['Avg_Price_USD_K'] = df_annual_avg['Price_USD'] / 1000
    X = df_annual_avg[['Year']] 
    Y = df_annual_avg['Avg_Price_USD_K']
    TRAIN_END_YEAR = 2021
    X_train = X[X['Year'] <= TRAIN_END_YEAR]
    Y_train = Y[X['Year'] <= TRAIN_END_YEAR]
    X_test = X[X['Year'] > TRAIN_END_YEAR]
    Y_test = Y[X['Year'] > TRAIN_END_YEAR]
    if X_test.empty:
        print(f"Error: Not enough test data (only data up to {TRAIN_END_YEAR}).")
        return None

    model_a = LinearRegression()
    model_a.fit(X_train, Y_train)
    Y_pred_a = model_a.predict(X_test)
    r2_a = r2_score(Y_test, Y_pred_a)
    mae_a = mean_absolute_error(Y_test, Y_pred_a) * 1000 

    print("Model A (Linear Regression) trained successfully.")
    print(f"R² Score: {r2_a:.4f} | MAE: ${mae_a:,.0f}")
    
    return X_train, Y_train, X_test, Y_test, model_a, r2_a, mae_a

input_file = 'BMW sales data (2010-2024) (1).csv'
data_and_model_a = prepare_data_and_train_model_a(input_file)

if data_and_model_a:
    X_train, Y_train, X_test, Y_test, model_a, r2_a, mae_a = data_and_model_a
    print("\nData and Model A ready for comparison.")

Model A (Linear Regression) trained successfully.
R² Score: 0.1305 | MAE: $828

Data and Model A ready for comparison.


In [3]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error 

def train_model_b_and_compare(X_train, Y_train, X_test, Y_test, r2_a, mae_a):
    
    model_b = RandomForestRegressor(n_estimators=58, random_state=11)
    model_b.fit(X_train, Y_train)
    Y_pred_b = model_b.predict(X_test)
    r2_b = r2_score(Y_test, Y_pred_b)
    mae_b = mean_absolute_error(Y_test, Y_pred_b) * 1000
    print("\nModel B (Random Forest) trained successfully.")
    print(f"R² Score: {r2_b:.4f} | MAE: ${mae_b:,.0f}")
    
    df_results = X_test.copy()
    df_results['Actual Price (k USD)'] = Y_test.round(2)
    df_results['LR Prediction (k USD)'] = pd.Series(model_a.predict(X_test)).round(2).values
    df_results['RF Prediction (k USD)'] = pd.Series(Y_pred_b).round(2).values

    print("\n1. Key Prediction Results (2022-2024 Test Set):\n")
    print(df_results.to_string(index=False))
    
    # Print Evaluation Metrics Comparison
    df_metrics = pd.DataFrame({
        'Model': ['Linear Regression (A)', 'Random Forest (B)'],
        'R-squared (R²)': [r2_a, r2_b],
        'MAE (USD)': [mae_a, mae_b]
    })
    
    print("\n2. Evaluation Metric Comparison:\n")
    print(df_metrics.to_string(index=False, float_format="%.4f"))
    
    return df_results, df_metrics

if 'data_and_model_a' in globals():
    results, metrics = train_model_b_and_compare(X_train, Y_train, X_test, Y_test, r2_a, mae_a)
else:
    print("Error: Run S5_DataPrep_ModelA.py first to initialize data and Model A.")


Model B (Random Forest) trained successfully.
R² Score: -0.1227 | MAE: $916

1. Key Prediction Results (2022-2024 Test Set):

 Year  Actual Price (k USD)  LR Prediction (k USD)  RF Prediction (k USD)
 2022                 74.90                  76.10                  75.74
 2023                 76.01                  76.19                  75.74
 2024                 77.37                  76.28                  75.74

2. Evaluation Metric Comparison:

                Model  R-squared (R²)  MAE (USD)
Linear Regression (A)          0.1305   827.7198
    Random Forest (B)         -0.1227   915.5109
